**© Copyright AIDENTIFY. All rights reserved.**

# Part 5 | Session 38: Agent AI 기술 스택과 LangGraph 기반 멀티 에이전트 워크플로우

이 노트북에서는 주요 Agent AI 프레임워크들을 비교하고, **LangGraph**를 활용한 그래프 기반 워크플로우 설계 및 멀티 에이전트 시스템 구축을 실습합니다.

## 환경 설정

In [ ]:
!pip install -q langgraph langchain langchain-openai langchain-community

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key를 입력하세요: ")

print("환경 설정 완료!")

---
## 1️⃣ Agent AI 기술 스택 개요

현재 Agent AI 생태계에는 다양한 프레임워크가 존재합니다. 각 프레임워크의 특성과 장단점을 비교해 봅시다.

### 주요 프레임워크 비교

| 프레임워크 | 핵심 특징 | 장점 | 단점 |
|-----------|----------|------|------|
| **LangChain** | LLM 애플리케이션 개발 프레임워크 | 풍부한 통합, 넓은 커뮤니티 | 추상화 과다, 러닝커브 |
| **LangGraph** | 그래프 기반 에이전트 오케스트레이션 | 순환 워크플로우, 상태 관리, 세밀한 제어 | LangChain 의존성 |
| **AG2 (AutoGen)** | 멀티 에이전트 대화 프레임워크 | 유연한 대화 패턴, Microsoft 지원 | 복잡한 설정 |
| **CrewAI** | 역할 기반 멀티 에이전트 | 직관적 API, 역할/목표 기반 설계 | 커스터마이징 제한 |

In [ ]:
# 프레임워크 비교 시각화
framework_comparison = {
    "프레임워크": ["LangChain", "LangGraph", "AG2 (AutoGen)", "CrewAI"],
    "출시": ["2022.10", "2024.01", "2023.09", "2023.12"],
    "핵심 개념": [
        "Chain, Agent, Tool",
        "StateGraph, Node, Edge",
        "ConversableAgent, GroupChat",
        "Agent, Task, Crew"
    ],
    "워크플로우 제어": ["순차적(Chain)", "그래프(순환 가능)", "대화 기반", "순차/병렬 Task"],
    "상태 관리": ["제한적", "내장(TypedDict/Pydantic)", "대화 히스토리", "제한적"],
    "적합 사례": [
        "RAG, 단순 에이전트",
        "복잡한 워크플로우, 멀티 에이전트",
        "에이전트 간 토론, 코드 생성",
        "팀 기반 자동화"
    ]
}

for i, name in enumerate(framework_comparison["프레임워크"]):
    print(f"\n{'='*60}")
    print(f"📌 {name} (출시: {framework_comparison['출시'][i]})")
    print(f"  핵심 개념: {framework_comparison['핵심 개념'][i]}")
    print(f"  워크플로우 제어: {framework_comparison['워크플로우 제어'][i]}")
    print(f"  상태 관리: {framework_comparison['상태 관리'][i]}")
    print(f"  적합 사례: {framework_comparison['적합 사례'][i]}")

### LangGraph를 선택하는 이유

- **순환 그래프 지원**: 에이전트가 결과를 검토하고 다시 실행할 수 있는 루프 구조
- **명시적 상태 관리**: `TypedDict` 또는 `Pydantic`으로 상태를 정의하여 투명한 데이터 흐름
- **세밀한 제어**: 각 노드(Node)와 엣지(Edge)를 직접 정의하여 워크플로우를 정밀하게 설계
- **Human-in-the-Loop**: 중간 단계에서 사람의 승인을 받는 패턴 지원
- **스트리밍 & 체크포인팅**: 실시간 출력과 상태 복원 기능 내장

---
## 2️⃣ LangGraph 소개: 그래프 기반 워크플로우

LangGraph는 **유한 상태 머신(Finite State Machine)** 개념을 기반으로 합니다.

### 핵심 구성 요소

```
┌─────────────────────────────────────────────┐
│              StateGraph                      │
│                                              │
│   [START] ──Edge──▶ [Node A]                │
│                        │                     │
│                   Conditional Edge           │
│                    ╱         ╲               │
│           [Node B]         [Node C]          │
│               │                │             │
│               └──────┬─────────┘             │
│                      ▼                       │
│                   [END]                      │
└─────────────────────────────────────────────┘
```

| 구성 요소 | 설명 |
|----------|------|
| **State** | 그래프를 통해 전달되는 데이터 구조 (TypedDict 또는 Pydantic 모델) |
| **StateGraph** | 상태를 기반으로 동작하는 그래프 컨테이너 |
| **Node** | 실제 작업을 수행하는 함수 (State를 받아 State를 반환) |
| **Edge** | 노드 간 연결, 조건부 분기 가능 |
| **START / END** | 그래프의 시작점과 종료점 |

In [ ]:
# LangGraph 핵심 모듈 임포트 및 확인
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
import operator

print("LangGraph 핵심 모듈 임포트 완료!")
print(f"  - StateGraph: 그래프 정의")
print(f"  - START: 시작 노드")
print(f"  - END: 종료 노드")
print(f"  - TypedDict: 상태 정의용")
print(f"  - Annotated + operator: 상태 리듀서 정의용")

In [ ]:
# 간단한 예제: 기본 그래프 구조 이해하기

# 1. 상태(State) 정의
class SimpleState(TypedDict):
    message: str
    step_count: int

# 2. 노드 함수 정의 (State를 받아 업데이트할 부분을 반환)
def step_a(state: SimpleState) -> dict:
    print(f"  [Node A] 입력 메시지: {state['message']}")
    return {"message": state["message"] + " → A 처리 완료", "step_count": state["step_count"] + 1}

def step_b(state: SimpleState) -> dict:
    print(f"  [Node B] 현재 메시지: {state['message']}")
    return {"message": state["message"] + " → B 처리 완료", "step_count": state["step_count"] + 1}

# 3. 그래프 구성
graph = StateGraph(SimpleState)
graph.add_node("step_a", step_a)
graph.add_node("step_b", step_b)

# 4. 엣지 연결
graph.add_edge(START, "step_a")
graph.add_edge("step_a", "step_b")
graph.add_edge("step_b", END)

# 5. 컴파일 및 실행
app = graph.compile()

result = app.invoke({"message": "시작", "step_count": 0})
print(f"\n최종 결과: {result}")

---
## 3️⃣ LangGraph 기본 실습: 간단한 챗봇 에이전트

LangGraph와 OpenAI LLM을 결합하여 대화형 챗봇 에이전트를 만들어 봅시다.

### 구현 흐름
1. **State 정의**: 대화 메시지 리스트를 상태로 관리
2. **챗봇 노드 추가**: LLM 호출을 수행하는 노드
3. **그래프 컴파일 및 실행**

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage
from typing import TypedDict, Annotated


# 1. 상태 정의 - add_messages 리듀서로 메시지 누적 관리
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]


# 2. LLM 초기화
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)


# 3. 챗봇 노드 정의
def chatbot_node(state: ChatState) -> dict:
    """LLM을 호출하여 응답을 생성하는 챗봇 노드"""
    response = llm.invoke(state["messages"])
    return {"messages": [response]}


# 4. 그래프 구성
chat_graph = StateGraph(ChatState)
chat_graph.add_node("chatbot", chatbot_node)
chat_graph.add_edge(START, "chatbot")
chat_graph.add_edge("chatbot", END)

# 5. 컴파일
chat_app = chat_graph.compile()

print("챗봇 그래프 구성 완료!")
print("구조: START → chatbot → END")

In [ ]:
# 챗봇 실행 테스트
response = chat_app.invoke({
    "messages": [HumanMessage(content="LangGraph가 뭔지 간단히 설명해줘.")]
})

# 마지막 AI 메시지 출력
print("사용자: LangGraph가 뭔지 간단히 설명해줘.")
print(f"\nAI: {response['messages'][-1].content}")

In [ ]:
# 멀티턴 대화 시뮬레이션
# add_messages 리듀서 덕분에 이전 대화 이력이 자동으로 유지됩니다.

conversation = {"messages": []}

user_inputs = [
    "나는 Python 개발자야. Agent AI에 대해 배우고 싶어.",
    "LangGraph의 장점이 뭐야?",
    "간단한 예제 코드를 보여줄 수 있어?"
]

for user_input in user_inputs:
    conversation["messages"].append(HumanMessage(content=user_input))
    result = chat_app.invoke(conversation)
    conversation = result  # 상태 업데이트
    
    print(f"\n{'='*60}")
    print(f"사용자: {user_input}")
    print(f"\nAI: {result['messages'][-1].content}")

---
## 4️⃣ Tool Calling + LangGraph: 도구 사용 에이전트

LLM이 직접 도구(Tool)를 호출하고, 그 결과를 활용하여 응답하는 **ReAct 패턴** 에이전트를 LangGraph로 구현합니다.

### 구현 흐름
```
START → agent(LLM) ──tool_calls 있음──▶ tools ──▶ agent(LLM)
                    │                                    │
                    └──tool_calls 없음──▶ END ◀──────────┘
```

순환(Cycle)이 발생하는 것이 핵심입니다 — LLM이 도구 호출이 필요 없다고 판단할 때까지 반복합니다.

In [ ]:
from langchain_core.tools import tool
import math
import json


# 도구(Tool) 정의
@tool
def search_web(query: str) -> str:
    """웹에서 정보를 검색합니다. 최신 정보나 사실 확인이 필요할 때 사용합니다."""
    # 시뮬레이션용 검색 결과
    mock_results = {
        "langgraph": "LangGraph는 LangChain 팀이 개발한 그래프 기반 에이전트 오케스트레이션 프레임워크입니다. 순환 그래프를 지원하며, 복잡한 멀티 에이전트 워크플로우를 구현할 수 있습니다.",
        "agent ai": "Agent AI는 LLM을 활용하여 자율적으로 작업을 수행하는 AI 시스템입니다. 도구 사용, 계획 수립, 자기 반성 등의 능력을 갖추고 있습니다.",
    }
    for key, value in mock_results.items():
        if key in query.lower():
            return value
    return f"'{query}'에 대한 검색 결과: 관련 정보를 찾았습니다. Agent AI 기술이 빠르게 발전하고 있습니다."


@tool
def calculator(expression: str) -> str:
    """수학 계산을 수행합니다. 사칙연산, 거듭제곱, 제곱근 등을 계산할 수 있습니다."""
    try:
        # 안전한 수학 함수들만 허용
        allowed = {"sqrt": math.sqrt, "pow": pow, "abs": abs, "round": round, "pi": math.pi}
        result = eval(expression, {"__builtins__": {}}, allowed)
        return f"계산 결과: {expression} = {result}"
    except Exception as e:
        return f"계산 오류: {str(e)}"


tools = [search_web, calculator]

print("정의된 도구들:")
for t in tools:
    print(f"  - {t.name}: {t.description}")

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_openai import ChatOpenAI
from typing import TypedDict, Annotated


# 1. 상태 정의
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


# 2. 도구가 바인딩된 LLM
llm_with_tools = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(tools)


# 3. 에이전트 노드: LLM 호출
def agent_node(state: AgentState) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


# 4. 조건부 엣지: 도구 호출 여부 판단
def should_use_tools(state: AgentState) -> str:
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return "end"


# 5. 그래프 구성
tool_graph = StateGraph(AgentState)

# 노드 추가
tool_graph.add_node("agent", agent_node)
tool_graph.add_node("tools", ToolNode(tools))

# 엣지 연결
tool_graph.add_edge(START, "agent")
tool_graph.add_conditional_edges(
    "agent",
    should_use_tools,
    {"tools": "tools", "end": END}
)
tool_graph.add_edge("tools", "agent")  # 도구 결과 → 다시 LLM으로

# 컴파일
tool_app = tool_graph.compile()

print("Tool Calling 에이전트 그래프 구성 완료!")
print("구조: START → agent ⇄ tools → END (순환 구조)")

In [ ]:
# Tool Calling 에이전트 테스트
from langchain_core.messages import HumanMessage

# 테스트 1: 검색 도구 사용
print("=" * 60)
print("테스트 1: 검색 도구 활용")
result1 = tool_app.invoke({
    "messages": [HumanMessage(content="LangGraph에 대해 검색해서 알려줘.")]
})
print(f"\nAI: {result1['messages'][-1].content}")

# 테스트 2: 계산 도구 사용
print(f"\n{'='*60}")
print("테스트 2: 계산 도구 활용")
result2 = tool_app.invoke({
    "messages": [HumanMessage(content="sqrt(144) + pow(3, 4)를 계산해줘.")]
})
print(f"\nAI: {result2['messages'][-1].content}")

# 테스트 3: 도구 없이 직접 답변
print(f"\n{'='*60}")
print("테스트 3: 도구 없이 직접 답변")
result3 = tool_app.invoke({
    "messages": [HumanMessage(content="안녕하세요! 오늘 기분이 어때요?")]
})
print(f"\nAI: {result3['messages'][-1].content}")

---
## 5️⃣ 멀티 에이전트 시스템: Supervisor 패턴

여러 전문 에이전트를 **Supervisor(감독자)**가 조율하는 패턴을 구현합니다.

### Supervisor 패턴 구조
```
                    ┌─────────────┐
    사용자 질문 ──▶ │  Supervisor  │
                    └──────┬──────┘
                    라우팅 │ 결정
              ┌───────────┼───────────┐
              ▼           ▼           ▼
         ┌────────┐ ┌────────┐ ┌────────┐
         │ 연구원  │ │ 작성자  │ │ 분석가  │
         └────────┘ └────────┘ └────────┘
              │           │           │
              └───────────┼───────────┘
                          ▼
                    ┌─────────────┐
                    │  Supervisor  │ ──▶ 최종 응답 또는 다음 에이전트
                    └─────────────┘
```

- **Supervisor**: 사용자 요청을 분석하고 적합한 에이전트에게 작업을 배분
- **전문 에이전트**: 각자의 역할에 맞는 작업 수행
- **순환 가능**: Supervisor가 결과를 검토하고 추가 작업을 지시할 수 있음

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from typing import TypedDict, Annotated, Literal


# 멀티 에이전트 상태 정의
class MultiAgentState(TypedDict):
    messages: Annotated[list, add_messages]
    next_agent: str  # 다음에 실행할 에이전트
    research_result: str  # 연구원 에이전트 결과
    analysis_result: str  # 분석가 에이전트 결과
    final_report: str  # 최종 보고서


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("멀티 에이전트 상태 및 LLM 초기화 완료!")

In [ ]:
# 각 에이전트 노드 정의

def supervisor_node(state: MultiAgentState) -> dict:
    """Supervisor: 작업을 분석하고 다음 에이전트를 결정"""
    system_msg = SystemMessage(content="""당신은 팀을 관리하는 Supervisor입니다.
사용자의 요청을 분석하여 다음 중 하나를 선택하세요:
- 'researcher': 정보 조사가 필요한 경우
- 'analyst': 데이터 분석이 필요한 경우  
- 'writer': 보고서 작성이 필요한 경우 (연구와 분석이 완료된 후)
- 'FINISH': 모든 작업이 완료된 경우

현재까지의 결과를 검토하고, 반드시 다음 에이전트 이름만 답변하세요.""")
    
    # 현재 상태 요약
    status = f"\n현재 상태 - 연구: {'완료' if state.get('research_result') else '미완료'}, 분석: {'완료' if state.get('analysis_result') else '미완료'}, 보고서: {'완료' if state.get('final_report') else '미완료'}"
    
    messages = [system_msg] + state["messages"] + [HumanMessage(content=status)]
    response = llm.invoke(messages)
    
    next_agent = response.content.strip().lower()
    # 유효한 에이전트 이름으로 매핑
    if "researcher" in next_agent or "연구" in next_agent:
        next_agent = "researcher"
    elif "analyst" in next_agent or "분석" in next_agent:
        next_agent = "analyst"
    elif "writer" in next_agent or "작성" in next_agent:
        next_agent = "writer"
    else:
        next_agent = "FINISH"
    
    print(f"  [Supervisor] 다음 에이전트: {next_agent}")
    return {"next_agent": next_agent}


def researcher_node(state: MultiAgentState) -> dict:
    """연구원: 주제에 대한 정보를 조사"""
    system_msg = SystemMessage(content="당신은 전문 연구원입니다. 주어진 주제에 대해 핵심 정보를 조사하여 3-5개의 핵심 포인트로 정리하세요. 한국어로 답변하세요.")
    messages = [system_msg] + state["messages"]
    response = llm.invoke(messages)
    
    print(f"  [연구원] 조사 완료")
    return {
        "research_result": response.content,
        "messages": [AIMessage(content=f"[연구원 보고] {response.content}")]
    }


def analyst_node(state: MultiAgentState) -> dict:
    """분석가: 연구 결과를 바탕으로 분석"""
    system_msg = SystemMessage(content="당신은 데이터 분석가입니다. 연구 결과를 바탕으로 트렌드, 장단점, 시사점을 분석하세요. 한국어로 답변하세요.")
    context = f"연구 결과: {state.get('research_result', '아직 없음')}"
    messages = [system_msg] + state["messages"] + [HumanMessage(content=context)]
    response = llm.invoke(messages)
    
    print(f"  [분석가] 분석 완료")
    return {
        "analysis_result": response.content,
        "messages": [AIMessage(content=f"[분석가 보고] {response.content}")]
    }


def writer_node(state: MultiAgentState) -> dict:
    """작성자: 연구+분석 결과를 종합하여 최종 보고서 작성"""
    system_msg = SystemMessage(content="당신은 전문 보고서 작성자입니다. 연구 결과와 분석 결과를 종합하여 간결한 최종 보고서를 작성하세요. 한국어로 답변하세요.")
    context = f"""연구 결과: {state.get('research_result', '없음')}
분석 결과: {state.get('analysis_result', '없음')}"""
    messages = [system_msg] + [HumanMessage(content=context)]
    response = llm.invoke(messages)
    
    print(f"  [작성자] 보고서 작성 완료")
    return {
        "final_report": response.content,
        "messages": [AIMessage(content=f"[최종 보고서] {response.content}")]
    }


print("모든 에이전트 노드 정의 완료!")

In [ ]:
# 멀티 에이전트 그래프 구성

def route_from_supervisor(state: MultiAgentState) -> str:
    """Supervisor의 결정에 따라 다음 노드로 라우팅"""
    return state["next_agent"]


# 그래프 생성
multi_graph = StateGraph(MultiAgentState)

# 노드 추가
multi_graph.add_node("supervisor", supervisor_node)
multi_graph.add_node("researcher", researcher_node)
multi_graph.add_node("analyst", analyst_node)
multi_graph.add_node("writer", writer_node)

# 엣지 연결
multi_graph.add_edge(START, "supervisor")

# Supervisor의 조건부 라우팅
multi_graph.add_conditional_edges(
    "supervisor",
    route_from_supervisor,
    {
        "researcher": "researcher",
        "analyst": "analyst",
        "writer": "writer",
        "FINISH": END
    }
)

# 각 에이전트 완료 후 Supervisor로 복귀
multi_graph.add_edge("researcher", "supervisor")
multi_graph.add_edge("analyst", "supervisor")
multi_graph.add_edge("writer", "supervisor")

# 컴파일
multi_app = multi_graph.compile()

print("멀티 에이전트 Supervisor 그래프 구성 완료!")
print("\n구조:")
print("  START → Supervisor ⇄ [Researcher, Analyst, Writer] → END")

In [ ]:
# 멀티 에이전트 실행
print("멀티 에이전트 팀에게 작업 요청 중...\n")

result = multi_app.invoke(
    {
        "messages": [HumanMessage(content="2025년 Agent AI 기술 트렌드에 대해 조사하고 분석 보고서를 작성해주세요.")],
        "next_agent": "",
        "research_result": "",
        "analysis_result": "",
        "final_report": ""
    },
    {"recursion_limit": 15}
)

print("\n" + "=" * 60)
print("최종 보고서:")
print("=" * 60)
print(result.get("final_report", "보고서가 생성되지 않았습니다."))

---
## 6️⃣ 실전 프로젝트 워크플로우: ML 파이프라인 설계

실제 ML 프로젝트에서 활용할 수 있는 **데이터 수집 → 정제 → 학습 → 평가 → 배포** 파이프라인을 LangGraph로 설계합니다.

### 파이프라인 구조
```
START → 데이터수집 → 데이터정제 → 품질검증 ──Pass──▶ 모델학습 → 모델평가
                        ▲          │                              │
                        └──Fail─────┘                     ┌──────┤
                                                    Pass  │  Fail│
                                                     ▼    │      ▼
                                                   배포   │  재학습
                                                     │    │      │
                                                     ▼    │      │
                                                   END ◀──┘──────┘
```

핵심 포인트: **조건부 분기**와 **루프**를 활용하여 품질 기준을 충족할 때까지 반복합니다.

In [ ]:
import random
from langgraph.graph import StateGraph, START, END
from typing import TypedDict


# 파이프라인 상태 정의
class PipelineState(TypedDict):
    data_collected: bool
    data_quality_score: float
    data_cleaned: bool
    model_trained: bool
    model_accuracy: float
    deployed: bool
    retry_count: int
    log: list[str]


# 각 파이프라인 단계 노드 정의
def collect_data(state: PipelineState) -> dict:
    """데이터 수집 단계"""
    print("  [1] 데이터 수집 중...")
    return {
        "data_collected": True,
        "log": state["log"] + ["데이터 수집 완료: 10,000건 수집"]
    }


def clean_data(state: PipelineState) -> dict:
    """데이터 정제 단계"""
    print("  [2] 데이터 정제 중...")
    quality = random.uniform(0.6, 1.0)
    return {
        "data_cleaned": True,
        "data_quality_score": round(quality, 2),
        "log": state["log"] + [f"데이터 정제 완료: 품질 점수 {quality:.2f}"]
    }


def validate_data(state: PipelineState) -> str:
    """데이터 품질 검증 (조건부 라우팅)"""
    score = state["data_quality_score"]
    print(f"  [3] 데이터 품질 검증: {score:.2f}")
    if score >= 0.8:
        print(f"      ✓ 품질 기준 통과 (>= 0.8)")
        return "train"
    else:
        print(f"      ✗ 품질 기준 미달 (< 0.8) → 재정제")
        return "reclean"


def train_model(state: PipelineState) -> dict:
    """모델 학습 단계"""
    print("  [4] 모델 학습 중...")
    accuracy = random.uniform(0.7, 0.98)
    return {
        "model_trained": True,
        "model_accuracy": round(accuracy, 3),
        "log": state["log"] + [f"모델 학습 완료: 정확도 {accuracy:.3f}"]
    }


def evaluate_model(state: PipelineState) -> str:
    """모델 평가 (조건부 라우팅)"""
    accuracy = state["model_accuracy"]
    print(f"  [5] 모델 평가: 정확도 {accuracy:.3f}")
    if accuracy >= 0.85:
        print(f"      ✓ 배포 기준 통과 (>= 0.85)")
        return "deploy"
    elif state["retry_count"] < 3:
        print(f"      ✗ 재학습 필요 (시도 {state['retry_count'] + 1}/3)")
        return "retrain"
    else:
        print(f"      ! 최대 재시도 초과 → 현재 모델로 배포")
        return "deploy"


def retrain_node(state: PipelineState) -> dict:
    """재학습 (retry_count 증가)"""
    return {"retry_count": state["retry_count"] + 1}


def deploy_model(state: PipelineState) -> dict:
    """모델 배포 단계"""
    print(f"  [6] 모델 배포 완료! (정확도: {state['model_accuracy']:.3f})")
    return {
        "deployed": True,
        "log": state["log"] + [f"모델 배포 완료: 정확도 {state['model_accuracy']:.3f}"]
    }


print("파이프라인 노드 함수 정의 완료!")

In [ ]:
# ML 파이프라인 그래프 구성
pipeline = StateGraph(PipelineState)

# 노드 추가
pipeline.add_node("collect", collect_data)
pipeline.add_node("clean", clean_data)
pipeline.add_node("train", train_model)
pipeline.add_node("retrain", retrain_node)
pipeline.add_node("deploy", deploy_model)

# 순차 엣지
pipeline.add_edge(START, "collect")
pipeline.add_edge("collect", "clean")

# 데이터 품질 검증 (조건부)
pipeline.add_conditional_edges(
    "clean",
    validate_data,
    {"train": "train", "reclean": "clean"}  # 미달 시 재정제 루프
)

# 모델 평가 (조건부)
pipeline.add_conditional_edges(
    "train",
    evaluate_model,
    {"deploy": "deploy", "retrain": "retrain"}
)

# 재학습 → 다시 학습으로
pipeline.add_edge("retrain", "train")

# 배포 → 종료
pipeline.add_edge("deploy", END)

# 컴파일
pipeline_app = pipeline.compile()

print("ML 파이프라인 그래프 구성 완료!")
print("\n구조:")
print("  START → 수집 → 정제 ⇄(품질검증) → 학습 ⇄(평가) → 배포 → END")

In [ ]:
# ML 파이프라인 실행
random.seed(42)  # 재현성을 위한 시드 설정

print("ML 파이프라인 실행 시작!")
print("=" * 60)

result = pipeline_app.invoke(
    {
        "data_collected": False,
        "data_quality_score": 0.0,
        "data_cleaned": False,
        "model_trained": False,
        "model_accuracy": 0.0,
        "deployed": False,
        "retry_count": 0,
        "log": []
    },
    {"recursion_limit": 20}
)

print(f"\n{'='*60}")
print("파이프라인 실행 로그:")
for entry in result["log"]:
    print(f"  - {entry}")

print(f"\n최종 상태:")
print(f"  배포 여부: {result['deployed']}")
print(f"  모델 정확도: {result['model_accuracy']:.3f}")
print(f"  재시도 횟수: {result['retry_count']}")

---
## 정리 및 핵심 요약

### 이번 세션에서 배운 내용

| 주제 | 핵심 내용 |
|------|----------|
| **Agent AI 기술 스택** | LangChain, LangGraph, AG2, CrewAI 각각의 특성과 적합한 사용 사례 |
| **LangGraph 기본 개념** | StateGraph, Node, Edge, 조건부 라우팅, 순환 그래프 |
| **챗봇 에이전트** | `add_messages` 리듀서를 활용한 대화 상태 관리 |
| **Tool Calling** | LLM이 도구를 자율적으로 호출하는 ReAct 패턴 구현 |
| **멀티 에이전트** | Supervisor 패턴으로 전문 에이전트 팀 조율 |
| **실전 파이프라인** | 조건부 분기와 루프를 활용한 ML 워크플로우 설계 |

### LangGraph 설계 팁

1. **상태를 먼저 설계하라**: `TypedDict`로 필요한 모든 데이터를 명시적으로 정의
2. **노드는 단일 책임 원칙**: 각 노드는 하나의 역할만 수행
3. **조건부 엣지를 활용하라**: 분기와 루프로 유연한 워크플로우 구성
4. **recursion_limit 설정**: 무한 루프 방지를 위해 반드시 설정
5. **Human-in-the-Loop 고려**: 중요한 의사결정에는 사람의 확인 단계 추가

In [ ]:
print("Session 38: Agent AI 기술 스택과 LangGraph 기반 워크플로우 실습 완료!")
print("\n다음 단계로 추천하는 학습 경로:")
print("  1. LangGraph의 체크포인팅과 상태 저장 기능 심화")
print("  2. Human-in-the-Loop 패턴 구현")
print("  3. LangGraph Studio를 활용한 시각적 디버깅")
print("  4. 프로덕션 환경을 위한 LangGraph Cloud 배포")
print("\n👉 다음 세션(Session 39, `39_data_pipeline_training.ipynb`): 프로젝트 데이터 파이프라인 구축")